In [22]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

In [23]:
data=pd.read_csv('all_tickets_processed.csv')

In [24]:
data.head()

,Document,Topic_group,Ticket Priority
0,connection with icon icon dear please setup ic...,Hardware,Medium
1,work experience user work experience user hi w...,Access,Medium
2,requesting for meeting requesting meeting hi p...,Hardware,Medium
3,reset passwords for external accounts re expir...,Access,Medium
4,mail verification warning hi has got attached ...,Miscellaneous,Medium


### Initial Data Exploration

We'll start by examining the distribution of `Topic_group` and `Ticket Priority` to understand the dataset's characteristics.

In [25]:
data.loc[0,'Document']

'connection with icon icon dear please setup icon per icon engineers please let other details needed thanks lead'

In [26]:
data['Topic_group'].value_counts()

Topic_group
Hardware                 13617
HR Support               10915
Access                    7125
Miscellaneous             7060
Storage                   2777
Purchase                  2464
Internal Project          2119
Administrative rights     1760
Name: count, dtype: int64

In [27]:
data['Ticket Priority'].value_counts()

Ticket Priority
Medium      20071
Low         11260
High        10652
Critical     5854
Name: count, dtype: int64

### Preprocessing Setup

This section defines the preprocessing steps required for our text data (`Document` column) and categorical target variables (`Topic_group`, `Ticket Priority`). We use `TfidfVectorizer` for text feature extraction and various encoders (`OrdinalEncoder`, `LabelEncoder`, `OneHotEncoder`) for the target variables.

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,OrdinalEncoder
def make_preprocessor():
    return ColumnTransformer(transformers=[
        ('vect', TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        ), 'Document')
    ])
priority_order = [["Low", "Medium", "High", "Critical"]]
oe = OrdinalEncoder(categories=priority_order)
ohe=OneHotEncoder()
le=LabelEncoder()



### Data Transformation

Here, we transform the target variables (`Ticket Priority` and `Topic_group`) using the defined encoders and separate them from the input features (`X`).

In [29]:
y=pd.DataFrame({'Ticket Priority':oe.fit_transform(data[['Ticket Priority']]).ravel(),
               "Topic_group"    : le.fit_transform(data["Topic_group"])})
X=data.drop(columns=['Ticket Priority','Topic_group'])

In [30]:
X.head()

,Document
0,connection with icon icon dear please setup ic...
1,work experience user work experience user hi w...
2,requesting for meeting requesting meeting hi p...
3,reset passwords for external accounts re expir...
4,mail verification warning hi has got attached ...


In [31]:
y.head()

,Ticket Priority,Topic_group
0,1.0,3
1,1.0,0
2,1.0,3
3,1.0,0
4,1.0,5


### Preparing Target Variables

We extract the encoded 'Ticket Priority' and 'Topic_group' into separate variables for individual model training.

In [32]:
X.head()

,Document
0,connection with icon icon dear please setup ic...
1,work experience user work experience user hi w...
2,requesting for meeting requesting meeting hi p...
3,reset passwords for external accounts re expir...
4,mail verification warning hi has got attached ...


In [33]:
y_priority=y['Ticket Priority']
y_grp=y['Topic_group']

### Data Splitting

The dataset is split into training and testing sets for both `Topic_group` and `Ticket Priority` prediction. Stratified sampling is used to maintain the original class distribution in both sets.

In [34]:
from sklearn.model_selection import train_test_split

X_grp_train,X_grp_test,y_grp_train,y_grp_test=train_test_split(X,y_grp,random_state=42,test_size=0.2,stratify=y["Topic_group"])
X_pri_train,X_pri_test,y_pri_train,y_pri_test=train_test_split(X,y_priority,random_state=42,test_size=0.2,stratify=y["Ticket Priority"])


In [35]:
y_grp_train.value_counts()

Topic_group
3    10893
2     8732
0     5700
5     5648
7     2222
6     1971
4     1695
1     1408
Name: count, dtype: int64

In [36]:
y_pri_train.value_counts()

Ticket Priority
1.0    16057
0.0     9008
2.0     8521
3.0     4683
Name: count, dtype: int64

### Model Pipelines Definition

We define two `Pipeline` objects: one for Logistic Regression and one for Random Forest. Each pipeline includes the `TfidfVectorizer` as a preprocessor followed by the respective classifier.

In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

In [38]:
pipe_lr=Pipeline([
    ('preprocessor',make_preprocessor()),
    ('classifier',LogisticRegression(max_iter=500))
])

pipe_rf=Pipeline([
    ('preprocessor',make_preprocessor()),
    ('classifier',RandomForestClassifier( class_weight="balanced",
        n_estimators=200,
        random_state=42,
        n_jobs=-1))
])

### Model Training and Evaluation

This section covers the training of the Logistic Regression model for 'Topic Group' prediction and the Random Forest model for 'Ticket Priority' prediction, along with their evaluation using classification reports.

**MODEL FOR PREDICTION OF TOPIC GROUP(LOGISTIC REGRESSTION)**

In [39]:
pipe_lr.fit(X_grp_train,y_grp_train)
y_grp_pre=pipe_lr.predict(X_grp_test)
print("REPORT OF LOGISTIC REGRESSION\n",classification_report(y_grp_test,y_grp_pre))

REPORT OF LOGISTIC REGRESSION
               precision    recall  f1-score   support

           0       0.91      0.88      0.89      1425
           1       0.88      0.67      0.76       352
           2       0.87      0.88      0.88      2183
           3       0.81      0.90      0.85      2724
           4       0.91      0.80      0.85       424
           5       0.83      0.83      0.83      1412
           6       0.98      0.86      0.91       493
           7       0.94      0.82      0.88       555

    accuracy                           0.86      9568
   macro avg       0.89      0.83      0.86      9568
weighted avg       0.86      0.86      0.86      9568



In [40]:
pipe_rf.fit(X_pri_train,y_pri_train)
y_pri_pre=pipe_rf.predict(X_pri_test)
print("REPORT OF RANDOM FOREST\n",classification_report(y_pri_test,y_pri_pre))


REPORT OF RANDOM FOREST
               precision    recall  f1-score   support

         0.0       0.88      0.73      0.80      2252
         1.0       0.77      0.91      0.84      4014
         2.0       0.80      0.77      0.78      2131
         3.0       1.00      0.78      0.87      1171

    accuracy                           0.82      9568
   macro avg       0.86      0.80      0.82      9568
weighted avg       0.83      0.82      0.82      9568



### Prediction Function

This helper function takes raw ticket text as input and uses the trained models to predict both the topic group and ticket priority, then prints the results in a human-readable format.

In [42]:
def predict(ticket_text):

    input_df = pd.DataFrame([{"Document": ticket_text}])

    grp_num  = pipe_lr.predict(input_df)[0]

    grp_word = le.inverse_transform([int(grp_num)])[0]

    pri_num  = pipe_rf.predict(input_df)[0]

    pri_word = oe.inverse_transform([[pri_num]])[0][0]

    print(oe.inverse_transform([[pri_num]]))

    print(f"Input    : {ticket_text}")
    print(f"Type     : {grp_word}")
    print(f"Priority : {pri_word}")
    print()

# ── Test ──────────────────────────────────────────────
predict("My laptop screen is broken and I cannot work")
predict("Please reset my password I am locked out")
predict("Need access to the finance module for new intern")
predict("Running low on mailbox storage please increase limit")

[['High']]
Input    : My laptop screen is broken and I cannot work
Type     : Hardware
Priority : High

[['High']]
Input    : Please reset my password I am locked out
Type     : Access
Priority : High

[['Low']]
Input    : Need access to the finance module for new intern
Type     : HR Support
Priority : Low

[['Medium']]
Input    : Running low on mailbox storage please increase limit
Type     : Storage
Priority : Medium



### Saving the trained models

To make the notebook production-ready, we will save the trained Logistic Regression model (for Topic Group prediction) and the Random Forest model (for Ticket Priority prediction) using `joblib`.

In [43]:
joblib.dump(pipe_lr, 'logistic_regression_model.pkl')

['logistic_regression_model.pkl']

In [44]:
joblib.dump(pipe_rf, 'random_forest_model.pkl')

['random_forest_model.pkl']

The models have been saved as `logistic_regression_model.pkl` and `random_forest_model.pkl`.